[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/38.%20The%20Automation%20Investment%20Analysis%20Problem/P38-Tier-5_executed.ipynb)

---


# 38. The Automation Investment Analysis Problem
## Tier 5: Integrated Digital Twin Simulation

### Goal

This notebook builds a small system-of-systems simulation for studying how a portfolio of automation projects may affect a container terminal over time. It represents terminal assets, operational process flows, investment implementation schedules, key performance indicators (KPIs), and simplified financial measures. It also compares investment scenarios and visualizes their simulated trajectories.

The method is a time-stepped simulation, not an optimization algorithm: it evaluates specified investment plans but does not search for an optimal portfolio. The simulation is useful for exploring assumptions and interactions; its outputs are not empirical forecasts.

### Inputs and outputs

**Inputs** are the terminal asset and process descriptions, project costs and effects, implementation durations, and simulation horizon. **Outputs** are monthly simulated states, annualized KPI and financial measures at selected times, scenario comparisons, and visualizations. Performance includes stochastic variation, so repeated runs can produce different trajectories.

### Model assumptions and scope

- Each process flow is represented by its listed required assets. Its performance is limited by the weakest required asset relative to that asset's initial performance.
- Asset efficiency and automation are bounded above by 1.0. Project effects accrue as implementation progresses; the monthly update applies only the newly completed fraction of each project.
- Asset performance has a normally distributed multiplicative variation, truncated to the range 0.8–1.2. No random seed is set, so results are not guaranteed to repeat exactly.
- Throughput and turnaround are linked through a simplified, dimensionless performance multiplier. Costs, revenue, energy use, reliability, and network effects are simplified representations, not a validated financial or operational model.
- Project costs are expressed in millions of dollars; financial metrics are expressed in dollars. Investment implementation timing is modeled, but project cash flows, discounting, taxes, demand constraints, network capacity, and uncertainty in project cost or duration are outside the model.
- The model does not integrate live terminal data, validate against observed operations, or represent all interactions in a real supply chain. The scenario results should therefore be interpreted as conditional illustrations of the stated assumptions.

### Simulation logic

Let \(a\in A\) index assets and \(f\in F\) index process flows. An asset's deterministic performance score is

$$
P_a = c_a e_a (1 + 0.2u_a),
$$

where \(c_a\) is capacity, \(e_a\in[0,1]\) is efficiency, and \(u_a\in[0,1]\) is automation level. The simulation multiplies this score by a bounded random factor. Because different asset types have different capacity units, system comparisons use each asset's performance relative to its own initial score:

$$
r_a = \frac{P_a}{P_a^{(0)}}, \qquad
r_f = \min_{a\in R_f} r_a,
$$

where \(P_a^{(0)}\) is the initial deterministic score and \(R_f\) is the set of assets required by process \(f\). The system throughput multiplier is the product of the average asset ratio and average process ratio:

$$
M = \left(\frac{1}{|A|}\sum_{a\in A} r_a\right)
    \left(\frac{1}{|F|}\sum_{f\in F} r_f\right).
$$

Thus, \(M=1\) represents initial expected performance, while investment-related changes and stochastic variation move it above or below that reference. Annual throughput is the baseline throughput multiplied by \(M\). Turnaround time responds to throughput improvement through a simplified relationship and is bounded below to remain positive.

For project \(j\), let \(q_{j,t}\in[0,1]\) be its completion fraction at month \(t\). The update applies the increment \(q_{j,t}-q_{j,t-1}\), so the same completed work is not applied repeatedly. Each simulated state records asset, process, KPI, and financial metrics.

This modeling sequence—represent the system, advance project implementation, simulate operational performance, and compare scenarios—is the central method used throughout the notebook. It supports what-if analysis, but it does not establish that a simulated improvement will occur in practice.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported for the digital twin simulation.")

In [ ]:
@dataclass
class PhysicalAsset:
    """A physical terminal asset and its operating characteristics."""

    id: str
    type: str
    capacity: float
    efficiency: float
    automation_level: float
    maintenance_cost: float
    energy_consumption: float


@dataclass
class ProcessFlow:
    """A terminal process and the assets required to perform it."""

    id: str
    name: str
    assets_required: List[str]
    processing_time: float
    variability: float
    bottleneck_potential: float


@dataclass
class InvestmentProject:
    """An investment project, with cost expressed in millions of dollars."""

    id: str
    name: str
    cost: float
    assets_affected: List[str]
    efficiency_improvement: float
    automation_increase: float
    implementation_time: int


@dataclass
class DigitalTwinState:
    """A simulated snapshot of the terminal at a particular time."""

    timestamp: datetime
    asset_states: Dict[str, Dict[str, float]]
    process_performance: Dict[str, float]
    kpi_metrics: Dict[str, float]
    financial_metrics: Dict[str, float]

print("Digital twin data structures defined.")

In [ ]:
def create_terminal_assets() -> List[PhysicalAsset]:
    """Create the initial terminal asset configuration."""
    assets = []

    for i in range(12):
        assets.append(PhysicalAsset(
            id=f"crane_{i}", type="crane", capacity=30.0,
            efficiency=0.75, automation_level=0.2,
            maintenance_cost=5000, energy_consumption=150
        ))

    for i in range(50):
        assets.append(PhysicalAsset(
            id=f"yard_crane_{i}", type="yard_crane", capacity=25.0,
            efficiency=0.70, automation_level=0.1,
            maintenance_cost=3000, energy_consumption=100
        ))

    for i in range(100):
        assets.append(PhysicalAsset(
            id=f"agv_{i}", type="vehicle", capacity=2.0,
            efficiency=0.65, automation_level=0.8,
            maintenance_cost=2000, energy_consumption=50
        ))

    for i in range(200):
        assets.append(PhysicalAsset(
            id=f"gate_{i}", type="gate", capacity=15.0,
            efficiency=0.80, automation_level=0.3,
            maintenance_cost=1000, energy_consumption=20
        ))

    return assets


def create_process_flows() -> List[ProcessFlow]:
    """Create terminal process flows and their required assets."""
    return [
        ProcessFlow(
            id="vessel_discharge", name="Vessel Discharge Operations",
            assets_required=["crane_0", "agv_0", "yard_crane_0"],
            processing_time=2.5, variability=0.3, bottleneck_potential=0.8
        ),
        ProcessFlow(
            id="vessel_load", name="Vessel Loading Operations",
            assets_required=["crane_1", "agv_1", "yard_crane_1"],
            processing_time=2.8, variability=0.35, bottleneck_potential=0.85
        ),
        ProcessFlow(
            id="truck_gate", name="Truck Gate Operations",
            assets_required=["gate_0"], processing_time=0.5,
            variability=0.4, bottleneck_potential=0.6
        ),
        ProcessFlow(
            id="yard_optimization", name="Yard Storage Optimization",
            assets_required=["yard_crane_2"], processing_time=1.2,
            variability=0.25, bottleneck_potential=0.7
        ),
    ]


def create_investment_projects() -> List[InvestmentProject]:
    """Create the specified automation investment portfolio."""
    return [
        InvestmentProject(
            id="autonomous_cranes", name="Autonomous Crane System",
            cost=35.0, assets_affected=[f"crane_{i}" for i in range(6)],
            efficiency_improvement=0.35, automation_increase=0.6,
            implementation_time=18
        ),
        InvestmentProject(
            id="ai_yard_management", name="AI-Powered Yard Management",
            cost=25.0, assets_affected=[f"yard_crane_{i}" for i in range(25)],
            efficiency_improvement=0.40, automation_increase=0.7,
            implementation_time=12
        ),
        InvestmentProject(
            id="predictive_maintenance", name="Predictive Maintenance AI",
            cost=15.0,
            assets_affected=[f"crane_{i}" for i in range(12)]
            + [f"yard_crane_{i}" for i in range(50)],
            efficiency_improvement=0.15, automation_increase=0.4,
            implementation_time=6
        ),
    ]


assets = create_terminal_assets()
process_flows = create_process_flows()
investment_projects = create_investment_projects()

print("Digital twin initialized with:")
print(f"- {len(assets)} physical assets")
print(f"- {len(process_flows)} process flows")
print(f"- {len(investment_projects)} investment projects")

## 1. Represent the terminal and calculate performance

The simulator stores assets and process flows by identifier so that a process can refer to the specific equipment it requires. Each asset's initial deterministic performance score is retained as a reference. Comparing an asset with its own reference makes the performance ratios dimensionless, so cranes, vehicles, and gates can be combined without averaging incompatible capacity units.

The bottleneck rule uses the minimum relative performance among a process's required assets. It expresses the idea that a chain of dependent operations may be constrained by its least capable required step. The calculation is intentionally simplified: it does not model queues, schedules, congestion, or interactions between individual jobs.

In [ ]:
class DigitalTwinSimulator:
    """Simulate terminal operations under a specified investment plan."""

    def __init__(self, assets: List[PhysicalAsset], flows: List[ProcessFlow]):
        # Copy inputs so that independent scenario simulators do not share mutable assets.
        self.assets = {asset.id: PhysicalAsset(**vars(asset)) for asset in assets}
        self.flows = {flow.id: ProcessFlow(**vars(flow)) for flow in flows}
        self.initial_asset_performance = {
            asset_id: self._deterministic_asset_performance(asset)
            for asset_id, asset in self.assets.items()
        }
        self.current_state = None
        self.simulation_history = []

        self.base_throughput = 2_500_000
        self.base_vessel_turnaround = 48.0
        self.base_operational_cost = 150_000_000.0

    @staticmethod
    def _deterministic_asset_performance(asset: PhysicalAsset) -> float:
        """Return expected asset performance before stochastic variation."""
        return asset.capacity * asset.efficiency * (1 + 0.2 * asset.automation_level)

    def apply_investment(
        self, project: InvestmentProject, completion_increment: float
    ) -> None:
        """Apply only the newly completed fraction of a project's effects."""
        if not 0.0 <= completion_increment <= 1.0:
            raise ValueError("Completion increment must be between 0 and 1.")

        for asset_id in project.assets_affected:
            if asset_id not in self.assets:
                continue

            asset = self.assets[asset_id]
            asset.efficiency = min(
                1.0,
                asset.efficiency
                + project.efficiency_improvement * completion_increment,
            )
            asset.automation_level = min(
                1.0,
                asset.automation_level
                + project.automation_increase * completion_increment,
            )
            asset.maintenance_cost *= (
                1 - 0.3 * project.automation_increase * completion_increment
            )

    def calculate_asset_performance(self, asset: PhysicalAsset) -> float:
        """Calculate an asset's capacity-adjusted performance with variation."""
        expected_performance = self._deterministic_asset_performance(asset)
        variation = np.random.normal(1.0, 0.05)
        return expected_performance * max(0.8, min(1.2, variation))

    def calculate_process_performance(self, flow: ProcessFlow) -> float:
        """Return a process bottleneck score relative to initial performance."""
        relative_performances = []
        for asset_id in flow.assets_required:
            if asset_id not in self.assets:
                raise ValueError(
                    f"Process '{flow.id}' requires unknown asset '{asset_id}'."
                )
            asset = self.assets[asset_id]
            initial_score = self.initial_asset_performance[asset_id]
            relative_performances.append(
                self.calculate_asset_performance(asset) / initial_score
            )

        if not relative_performances:
            return 1.0

        bottleneck_ratio = min(relative_performances)
        variability_factor = 1 - 0.1 * flow.variability
        return bottleneck_ratio * variability_factor

    def calculate_system_kpis(self) -> Dict[str, float]:
        """Calculate throughput, turnaround, cost, energy, and reliability KPIs."""
        asset_ratios = [
            self.calculate_asset_performance(asset)
            / self.initial_asset_performance[asset_id]
            for asset_id, asset in self.assets.items()
        ]
        process_ratios = [
            self.calculate_process_performance(flow) / (1 - 0.1 * flow.variability)
            for flow in self.flows.values()
        ]

        throughput_multiplier = float(np.mean(asset_ratios) * np.mean(process_ratios))
        annual_throughput = self.base_throughput * throughput_multiplier

        turnaround_factor = max(
            0.1, 1 - 0.5 * (throughput_multiplier - 1)
        )
        vessel_turnaround = self.base_vessel_turnaround * turnaround_factor

        automation_level = float(
            np.mean([asset.automation_level for asset in self.assets.values()])
        )
        annual_operational_cost = self.base_operational_cost * (
            1 - 0.15 * automation_level
        )
        total_power = sum(
            asset.energy_consumption for asset in self.assets.values()
        )
        annual_energy_consumption = total_power * 8760 * (
            1 - 0.25 * automation_level
        )
        system_reliability = 0.85 + 0.1 * automation_level

        return {
            "annual_throughput": annual_throughput,
            "vessel_turnaround_hours": vessel_turnaround,
            "annual_operational_cost": annual_operational_cost,
            "annual_energy_consumption": annual_energy_consumption,
            "system_reliability": system_reliability,
            "throughput_multiplier": throughput_multiplier,
        }

    def calculate_financial_metrics(
        self, kpis: Dict[str, float]
    ) -> Dict[str, float]:
        """Calculate simplified annual revenue, costs, and net profit."""
        annual_revenue = kpis["annual_throughput"] * 50.0
        annual_operating_profit = (
            annual_revenue - kpis["annual_operational_cost"]
        )
        annual_energy_cost = kpis["annual_energy_consumption"] * 0.15
        annual_maintenance_cost = sum(
            asset.maintenance_cost for asset in self.assets.values()
        ) * 12
        annual_net_profit = (
            annual_operating_profit
            - annual_energy_cost
            - annual_maintenance_cost
        )

        return {
            "annual_revenue": annual_revenue,
            "annual_operating_profit": annual_operating_profit,
            "annual_energy_cost": annual_energy_cost,
            "annual_maintenance_cost": annual_maintenance_cost,
            "annual_net_profit": annual_net_profit,
        }

    def simulate_step(self, timestamp: datetime) -> DigitalTwinState:
        """Generate and store one simulated system state."""
        kpis = self.calculate_system_kpis()
        financials = self.calculate_financial_metrics(kpis)
        process_performance = {
            flow_id: self.calculate_process_performance(flow)
            for flow_id, flow in self.flows.items()
        }
        asset_states = {
            asset_id: {
                "performance": self.calculate_asset_performance(asset),
                "efficiency": asset.efficiency,
                "automation_level": asset.automation_level,
                "maintenance_cost": asset.maintenance_cost,
            }
            for asset_id, asset in self.assets.items()
        }

        state = DigitalTwinState(
            timestamp=timestamp,
            asset_states=asset_states,
            process_performance=process_performance,
            kpi_metrics=kpis,
            financial_metrics=financials,
        )
        self.current_state = state
        self.simulation_history.append(state)
        return state

print("Digital twin simulator implementation complete.")

## 2. Establish baseline performance

The baseline is simulated before any projects are applied. It provides the reference values used to describe subsequent changes. Since asset performance contains random variation, the baseline is one stochastic observation rather than a perfectly fixed expectation. The initial asset performance reference used in dimensionless ratios remains fixed for the life of each simulator.

In [ ]:
simulator = DigitalTwinSimulator(assets, process_flows)
baseline_state = simulator.simulate_step(datetime.now())

print("BASELINE PERFORMANCE (Before Automation Investments)")
print("=" * 60)
print(
    f"Annual Throughput: "
    f"{baseline_state.kpi_metrics['annual_throughput']:,.0f} TEU"
)
print(
    f"Vessel Turnaround: "
    f"{baseline_state.kpi_metrics['vessel_turnaround_hours']:.1f} hours"
)
print(
    f"Annual Operational Cost: "
    f"${baseline_state.kpi_metrics['annual_operational_cost']:,.0f}"
)
print(
    f"System Reliability: "
    f"{baseline_state.kpi_metrics['system_reliability']:.1%}"
)
print(
    f"Annual Revenue: "
    f"${baseline_state.financial_metrics['annual_revenue']:,.0f}"
)
print(
    f"Annual Net Profit: "
    f"${baseline_state.financial_metrics['annual_net_profit']:,.0f}"
)

baseline_automation = float(
    np.mean([asset.automation_level for asset in simulator.assets.values()])
)
baseline_efficiency = float(
    np.mean([asset.efficiency for asset in simulator.assets.values()])
)
print("\nBaseline Asset Metrics:")
print(f"Average Automation Level: {baseline_automation:.1%}")
print(f"Average Efficiency: {baseline_efficiency:.1%}")

baseline_kpis = baseline_state.kpi_metrics.copy()
baseline_financials = baseline_state.financial_metrics.copy()

## 3. Simulate project implementation over time

The implementation schedule starts projects six months apart. At each month, the simulator calculates each project's completion fraction and applies only the change since the preceding month. This discrete-time update preserves the intended gradual implementation while avoiding repeated application of work already completed.

A state at a given month reports annualized KPI and financial measures at that simulated point; these are not cumulative totals over the entire simulation horizon. Project costs are shown in millions of dollars, while the simulator's annual revenue, cost, and profit measures are in dollars.

In [ ]:
def simulate_investment_implementation(
    simulator: DigitalTwinSimulator,
    projects: List[InvestmentProject],
    simulation_months: int = 60,
    verbose: bool = True,
) -> List[DigitalTwinState]:
    """Simulate scheduled project implementation month by month."""
    if simulation_months < 0:
        raise ValueError("Simulation horizon must be non-negative.")

    simulation_results = []
    start_time = datetime.now()
    project_schedule = {}
    previous_progress = {project.id: 0.0 for project in projects}

    for index, project in enumerate(projects):
        if project.implementation_time <= 0:
            raise ValueError("Project implementation time must be positive.")
        start_month = index * 6
        project_schedule[project.id] = {
            "start_month": start_month,
            "end_month": start_month + project.implementation_time,
            "project": project,
        }

    if verbose:
        print(f"Simulating {len(projects)} projects over {simulation_months} months...")
        print(f"Total Investment: ${sum(p.cost for p in projects):.1f}M")

    for month in range(simulation_months + 1):
        current_time = start_time + timedelta(days=30 * month)

        for project_id, schedule in project_schedule.items():
            project = schedule["project"]
            if schedule["start_month"] <= month <= schedule["end_month"]:
                progress = min(
                    1.0,
                    (month - schedule["start_month"])
                    / project.implementation_time,
                )
                increment = progress - previous_progress[project_id]
                if increment > 0:
                    simulator.apply_investment(project, increment)
                    previous_progress[project_id] = progress

        state = simulator.simulate_step(current_time)
        simulation_results.append(state)

        if verbose and month % 12 == 0:
            year = month // 12
            print(
                f"Year {year}: Throughput = "
                f"{state.kpi_metrics['annual_throughput']:,.0f} TEU, "
                f"Net Profit = "
                f"${state.financial_metrics['annual_net_profit']:,.0f}"
            )

    return simulation_results


simulation_results = simulate_investment_implementation(
    simulator, investment_projects, simulation_months=60
)
final_state = simulation_results[-1]

throughput_improvement = (
    (final_state.kpi_metrics["annual_throughput"]
     - baseline_kpis["annual_throughput"])
    / baseline_kpis["annual_throughput"]
    * 100
)
turnaround_reduction = (
    (baseline_kpis["vessel_turnaround_hours"]
     - final_state.kpi_metrics["vessel_turnaround_hours"])
    / baseline_kpis["vessel_turnaround_hours"]
    * 100
)
operational_cost_reduction = (
    (baseline_kpis["annual_operational_cost"]
     - final_state.kpi_metrics["annual_operational_cost"])
    / baseline_kpis["annual_operational_cost"]
    * 100
)
profit_improvement = (
    (final_state.financial_metrics["annual_net_profit"]
     - baseline_financials["annual_net_profit"])
    / abs(baseline_financials["annual_net_profit"])
    * 100
)

direct_profit_increase = (
    final_state.financial_metrics["annual_net_profit"]
    - baseline_financials["annual_net_profit"]
)
network_effect_value = direct_profit_increase * 0.15

print("\n" + "=" * 60)
print("INVESTMENT SIMULATION RESULTS")
print("=" * 60)
print("\nFinal Performance (After 5 Years):")
print(
    f"Annual Throughput: "
    f"{final_state.kpi_metrics['annual_throughput']:,.0f} TEU"
)
print(
    f"Vessel Turnaround: "
    f"{final_state.kpi_metrics['vessel_turnaround_hours']:.1f} hours"
)
print(
    f"Annual Operational Cost: "
    f"${final_state.kpi_metrics['annual_operational_cost']:,.0f}"
)
print(f"System Reliability: {final_state.kpi_metrics['system_reliability']:.1%}")
print(
    f"Annual Revenue: "
    f"${final_state.financial_metrics['annual_revenue']:,.0f}"
)
print(
    f"Annual Net Profit: "
    f"${final_state.financial_metrics['annual_net_profit']:,.0f}"
)
print("\nChanges Relative to the Simulated Baseline:")
print(f"- Throughput change: {throughput_improvement:.1f}%")
print(f"- Turnaround time reduction: {turnaround_reduction:.1f}%")
print(f"- Operational cost reduction: {operational_cost_reduction:.1f}%")
print(f"- Net profit change: {profit_improvement:.1f}%")
print("\nIllustrative Network Effects Estimate:")
print(f"- Direct annual profit change: ${direct_profit_increase:,.0f}")
print(f"- Assumed additional value (15%): ${network_effect_value:,.0f}")
print(
    f"- Combined annual value estimate: "
    f"${direct_profit_increase + network_effect_value:,.0f}"
)
print(
    "The network effect is an assumed uplift applied to the simulated "
    "profit change, not an independently modeled or measured effect."
)

## 4. Inspect system performance over time

The first figure follows annualized throughput, annualized net profit, turnaround time, and system reliability across the monthly simulation steps. A horizontal reference line indicates the simulated baseline. The shaded region highlights the difference from that reference; it is not a confidence interval. Since the performance model includes random variation, month-to-month movement does not necessarily indicate a change in project completion.

In [ ]:
months = np.arange(len(simulation_results))
throughput_data = [
    state.kpi_metrics["annual_throughput"] / 1e6
    for state in simulation_results
]
profit_data = [
    state.financial_metrics["annual_net_profit"] / 1e6
    for state in simulation_results
]
turnaround_data = [
    state.kpi_metrics["vessel_turnaround_hours"]
    for state in simulation_results
]
reliability_data = [
    state.kpi_metrics["system_reliability"] * 100
    for state in simulation_results
]

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

baseline_throughput = baseline_kpis["annual_throughput"] / 1e6
ax1.plot(months, throughput_data, linewidth=2.5, label="Investment portfolio")
ax1.axhline(
    baseline_throughput, color="red", linestyle="--", label="Baseline"
)
ax1.fill_between(months, baseline_throughput, throughput_data, alpha=0.2)
ax1.set_xlabel("Simulation month")
ax1.set_ylabel("Annual throughput (million TEU)")
ax1.set_title("Annualized Throughput")
ax1.grid(True, alpha=0.3)
ax1.legend()

baseline_profit = baseline_financials["annual_net_profit"] / 1e6
ax2.plot(months, profit_data, color="green", linewidth=2.5, label="Net profit")
ax2.axhline(baseline_profit, color="red", linestyle="--", label="Baseline")
ax2.fill_between(months, baseline_profit, profit_data, alpha=0.2)
ax2.set_xlabel("Simulation month")
ax2.set_ylabel("Annual net profit (million dollars)")
ax2.set_title("Annualized Financial Performance")
ax2.grid(True, alpha=0.3)
ax2.legend()

ax3.plot(months, turnaround_data, color="darkorange", linewidth=2.5)
ax3.axhline(
    baseline_kpis["vessel_turnaround_hours"],
    color="red", linestyle="--", label="Baseline"
)
ax3.set_xlabel("Simulation month")
ax3.set_ylabel("Vessel turnaround time (hours)")
ax3.set_title("Vessel Turnaround Time")
ax3.grid(True, alpha=0.3)
ax3.legend()

ax4.plot(months, reliability_data, color="purple", linewidth=2.5)
ax4.axhline(85, color="red", linestyle="--", label="Model reference")
ax4.set_xlabel("Simulation month")
ax4.set_ylabel("System reliability (%)")
ax4.set_title("System Reliability")
ax4.set_ylim(0, 100)
ax4.grid(True, alpha=0.3)
ax4.legend()

fig.tight_layout()
plt.show()
print("Monthly system performance visualized.")

## 5. Examine assets, process bottlenecks, and project estimates

The asset summaries show average simulated performance and the modeled automation and efficiency levels by asset type. The process heatmap displays the final simulated process scores; these scores include the process variability factor, so their absolute values are not identical to the dimensionless ratios used in the system throughput multiplier.

The project ROI chart is a simple allocation estimate. It allocates the portfolio's simulated annual profit change in proportion to project cost, assumes that this annual change persists for five years, and compares the resulting cumulative amount with the project's cost in dollars. It is not a causal estimate of each project's individual contribution, and it does not discount cash flows.

In [ ]:
asset_groups = {}
for asset in simulator.assets.values():
    asset_groups.setdefault(asset.type, []).append(asset)

type_performance = {}
for asset_type, type_assets in asset_groups.items():
    performance_values = [
        simulator.calculate_asset_performance(asset) for asset in type_assets
    ]
    type_performance[asset_type] = {
        "average_performance": float(np.mean(performance_values)),
        "average_automation": float(
            np.mean([asset.automation_level for asset in type_assets])
        ),
        "average_efficiency": float(
            np.mean([asset.efficiency for asset in type_assets])
        ),
        "count": len(type_assets),
    }

type_names = list(type_performance)
performance_values = [
    type_performance[name]["average_performance"] for name in type_names
]
automation_values = [
    type_performance[name]["average_automation"] * 100 for name in type_names
]
efficiency_values = [
    type_performance[name]["average_efficiency"] * 100 for name in type_names
]

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

bars = ax1.bar(type_names, performance_values, alpha=0.85)
ax1.set_ylabel("Average performance (capacity units per hour)")
ax1.set_title("Simulated Asset Performance by Type")
ax1.tick_params(axis="x", rotation=35)
ax1.grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, performance_values):
    ax1.annotate(
        f"{value:.1f}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 3), textcoords="offset points", ha="center"
    )

x = np.arange(len(type_names))
width = 0.35
ax2.bar(x - width / 2, automation_values, width, label="Automation level")
ax2.bar(x + width / 2, efficiency_values, width, label="Efficiency")
ax2.set_ylabel("Average level (%)")
ax2.set_title("Automation and Efficiency by Asset Type")
ax2.set_xticks(x)
ax2.set_xticklabels(type_names, rotation=35, ha="right")
ax2.legend()
ax2.grid(True, axis="y", alpha=0.3)

process_names = list(final_state.process_performance)
process_values = [final_state.process_performance[name] for name in process_names]
image = ax3.imshow(
    np.array(process_values).reshape(-1, 1),
    cmap="RdYlGn", aspect="auto"
)
ax3.set_xticks([0])
ax3.set_xticklabels(["Simulated score"])
ax3.set_yticks(range(len(process_names)))
ax3.set_yticklabels([name.replace("_", " ").title() for name in process_names])
ax3.set_title("Final Process Performance")
fig.colorbar(image, ax=ax3, label="Process score")
for row, value in enumerate(process_values):
    ax3.text(0, row, f"{value:.2f}", ha="center", va="center")

investment_costs = [project.cost for project in investment_projects]
project_names = [project.name.replace(" ", "\n") for project in investment_projects]
project_rois = []
for project in investment_projects:
    allocated_share = project.cost / sum(investment_costs)
    allocated_annual_change = direct_profit_increase * allocated_share
    roi = (
        allocated_annual_change * 5 / (project.cost * 1e6) * 100
    )
    project_rois.append(roi)

ax4_twin = ax4.twinx()
ax4.bar(project_names, investment_costs, color="orange", alpha=0.7,
       label="Investment cost")
ax4_twin.plot(
    project_names, project_rois, "bo-", linewidth=2, label="Estimated 5-year ROI"
)
ax4.set_ylabel("Investment cost (million dollars)")
ax4_twin.set_ylabel("Estimated 5-year ROI (%)")
ax4.set_title("Investment Cost and Allocated ROI Estimate")
ax4.tick_params(axis="x", rotation=25)
ax4.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()
print("Asset, process, and investment summaries visualized.")

## 6. Compare what-if scenarios

Each scenario starts from a separate copy of the initial asset configuration. This is necessary because investments modify asset characteristics; sharing mutable asset objects would allow one scenario to affect another. The scenario set includes the specified portfolio, an accelerated schedule with a cost premium, a phased portfolio using the first two projects, and a portfolio with an additional gate automation project.

The reported five-year ROI is calculated as five times the simulated annual net profit change relative to the baseline, divided by total investment in dollars. This is a simplified, undiscounted comparison, not a full investment appraisal. Scenario outputs are stochastic and should be compared as conditional simulation results rather than guaranteed rankings.

In [ ]:
def run_scenario_analysis(
    base_assets: List[PhysicalAsset],
    base_flows: List[ProcessFlow],
    base_projects: List[InvestmentProject],
    baseline_net_profit: float,
) -> Dict[str, Dict[str, object]]:
    """Run the portfolio, accelerated, phased, and expanded scenarios."""
    scenarios = {}

    simulator1 = DigitalTwinSimulator(base_assets, base_flows)
    results1 = simulate_investment_implementation(
        simulator1, base_projects, simulation_months=60, verbose=False
    )
    scenarios["baseline"] = {
        "final_state": results1[-1],
        "total_investment": sum(project.cost for project in base_projects),
    }

    accelerated_projects = [
        InvestmentProject(
            id=project.id,
            name=project.name,
            cost=project.cost * 1.2,
            assets_affected=project.assets_affected.copy(),
            efficiency_improvement=project.efficiency_improvement,
            automation_increase=project.automation_increase,
            implementation_time=max(1, project.implementation_time // 2),
        )
        for project in base_projects
    ]
    simulator2 = DigitalTwinSimulator(base_assets, base_flows)
    results2 = simulate_investment_implementation(
        simulator2, accelerated_projects, simulation_months=36, verbose=False
    )
    scenarios["accelerated"] = {
        "final_state": results2[-1],
        "total_investment": sum(
            project.cost for project in accelerated_projects
        ),
    }

    phased_projects = base_projects[:2]
    simulator3 = DigitalTwinSimulator(base_assets, base_flows)
    results3 = simulate_investment_implementation(
        simulator3, phased_projects, simulation_months=60, verbose=False
    )
    scenarios["phased"] = {
        "final_state": results3[-1],
        "total_investment": sum(project.cost for project in phased_projects),
    }

    full_projects = base_projects + [
        InvestmentProject(
            id="full_gate_automation",
            name="Full Gate Automation",
            cost=20.0,
            assets_affected=[f"gate_{i}" for i in range(100)],
            efficiency_improvement=0.50,
            automation_increase=0.8,
            implementation_time=24,
        )
    ]
    simulator4 = DigitalTwinSimulator(base_assets, base_flows)
    results4 = simulate_investment_implementation(
        simulator4, full_projects, simulation_months=72, verbose=False
    )
    scenarios["full_automation"] = {
        "final_state": results4[-1],
        "total_investment": sum(project.cost for project in full_projects),
    }

    for scenario in scenarios.values():
        state = scenario["final_state"]
        investment_dollars = scenario["total_investment"] * 1e6
        annual_profit_change = (
            state.financial_metrics["annual_net_profit"] - baseline_net_profit
        )
        scenario["roi_5year"] = (
            annual_profit_change * 5 / investment_dollars * 100
        )

    return scenarios


scenarios = run_scenario_analysis(
    assets, process_flows, investment_projects,
    baseline_financials["annual_net_profit"]
)

print("SCENARIO COMPARISON RESULTS")
print("=" * 60)
for scenario_name, scenario in scenarios.items():
    state = scenario["final_state"]
    print(f"\n{scenario_name.replace('_', ' ').title()} Scenario:")
    print(f"  Total investment: ${scenario['total_investment']:.1f}M")
    print(
        f"  Annual throughput: "
        f"{state.kpi_metrics['annual_throughput']:,.0f} TEU"
    )
    print(
        f"  Annual net profit: "
        f"${state.financial_metrics['annual_net_profit']:,.0f}"
    )
    print(f"  System reliability: {state.kpi_metrics['system_reliability']:.1%}")
    print(f"  Simplified five-year ROI: {scenario['roi_5year']:.1f}%")

scenario_names = list(scenarios)
throughputs = [
    scenarios[name]["final_state"].kpi_metrics["annual_throughput"] / 1e6
    for name in scenario_names
]
profits = [
    scenarios[name]["final_state"].financial_metrics["annual_net_profit"] / 1e6
    for name in scenario_names
]
investments = [scenarios[name]["total_investment"] for name in scenario_names]
reliabilities = [
    scenarios[name]["final_state"].kpi_metrics["system_reliability"] * 100
    for name in scenario_names
]
rois = [scenarios[name]["roi_5year"] for name in scenario_names]

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

ax1.bar(scenario_names, throughputs, color="skyblue", alpha=0.85)
ax1.axhline(
    baseline_kpis["annual_throughput"] / 1e6,
    color="red", linestyle="--", label="Simulated baseline"
)
ax1.set_ylabel("Annual throughput (million TEU)")
ax1.set_title("Throughput by Scenario")
ax1.tick_params(axis="x", rotation=25)
ax1.grid(True, axis="y", alpha=0.3)
ax1.legend()

ax2.bar(scenario_names, profits, color="lightgreen", alpha=0.85)
ax2.axhline(
    baseline_financials["annual_net_profit"] / 1e6,
    color="red", linestyle="--", label="Simulated baseline"
)
ax2.set_ylabel("Annual net profit (million dollars)")
ax2.set_title("Net Profit by Scenario")
ax2.tick_params(axis="x", rotation=25)
ax2.grid(True, axis="y", alpha=0.3)
ax2.legend()

ax3_twin = ax3.twinx()
ax3.bar(scenario_names, investments, color="orange", alpha=0.7,
        label="Investment")
ax3_twin.plot(
    scenario_names, rois, "ro-", linewidth=2, label="Simplified five-year ROI"
)
ax3.set_ylabel("Investment (million dollars)")
ax3_twin.set_ylabel("Simplified five-year ROI (%)")
ax3.set_title("Investment and ROI Estimate")
ax3.tick_params(axis="x", rotation=25)
ax3.grid(True, axis="y", alpha=0.3)

categories = ["Throughput", "Profit", "Reliability", "Cost efficiency"]
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]
max_investment = max(investments)
for index, scenario_name in enumerate(scenario_names):
    values = [
        throughputs[index] / max(throughputs) * 100,
        profits[index] / max(profits) * 100,
        reliabilities[index],
        (max_investment - investments[index]) / max_investment * 100,
    ]
    values += values[:1]
    ax4.plot(angles, values, marker="o", linewidth=2, label=scenario_name)
    ax4.fill(angles, values, alpha=0.12)
ax4.set_xticks(angles[:-1])
ax4.set_xticklabels(categories)
ax4.set_ylim(0, 100)
ax4.set_title("Normalized Scenario Comparison")
ax4.set_ylabel("Normalized score (%)")
ax4.legend()
ax4.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()
print("Scenario comparison complete.")

## Interpretation and limitations

Use the time-series plots to examine when modeled project effects enter the system and how the simplified KPIs respond. Use the scenario comparison to inspect trade-offs among throughput, profit, reliability, and investment cost. The radar chart normalizes throughput and profit relative to the largest scenario value, displays reliability as a percentage, and assigns a higher cost-efficiency score to lower investment cost. Its axes therefore have different underlying interpretations despite sharing a 0–100 scale.

A simulated outcome is conditional on the input data, equations, and random variation. The model's throughput relationship, financial assumptions, and network-effect estimate are illustrative; they should not be treated as validated predictions or evidence of causal project impacts. In practical use, these assumptions would need operational data, calibration, and validation. Natural extensions include representing demand limits, queues, project cash-flow timing, and uncertainty in project estimates while retaining the same time-stepped simulation framework.